# Feature Engineering  

This notebook uses the cleaned data from the folder data/processed and creates additional features.  

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np 
from ydata_profiling import ProfileReport
from c08_farming_exit import config, features, data_cleaning, mappings, feature_engineering

In [2]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

## 1. Import processed data

In [3]:
df = pd.read_csv(config.PROCESSED_DATA_DIR / "clean_data.csv")

## 2. Employment features

### 2.1 Employment categories

In [4]:
#EMPLOYMENT CATEGORIES
conditions = [
    (df["farm_empl_last_12_months"] == 1) & (df["empl_type"].isnull()),
    (df["farm_empl_last_12_months"] == 1) & (df["empl_type"].notnull()),
    (df["farm_empl_last_12_months"] == 0) & (df["empl_type"].notnull()),
]
choices = ["only_farm", "hybrid", "fully_off_farm"]

df["empl_category"] = np.select(conditions, choices, default=None)

### 2.2 Work hours per year - absolute numbers

In [5]:
#FARMING
df["cash_crop_hours_per_year"] = np.where(
    df["farm_empl_last_12_months"] == 1,
    (df["farm_empl_cash_crops_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_cash_crops_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_cash_crops_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["farm_empl_cash_crops_hours_per_day"],
    np.nan
)

df["food_crop_hours_per_year"] = np.where(
    df["farm_empl_last_12_months"] == 1,
    (df["farm_empl_food_crops_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_food_crops_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_food_crops_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["farm_empl_food_crops_hours_per_day"],
    np.nan
)

# No filtering here: livestock owners that don't do crop farming might not claim that they have worked on the farm. 
df["livestock_hours_per_year"] = (
    (df["farm_empl_livestock_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_livestock_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_livestock_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["farm_empl_livestock_hours_per_day"]
)

df["farm_empl_hours_per_year"] = df[["cash_crop_hours_per_year", "food_crop_hours_per_year", "livestock_hours_per_year"]].sum(axis=1, min_count=1)

In [6]:
#SELF-EMPLOYMENT
df["self_empl_hours_per_year"] = np.where(
    df["empl_type"] == "Self-employed/own business",
    (df["self_empl_duration_rainy_season_in_months_last_12_months"]
     + df["self_empl_duration_dry_season_in_months_last_12_months"])
    * (df["self_empl_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["self_empl_hours_per_day"],
    np.nan
)

In [7]:
#PERMANENT WAGE EMPLOYMENT
df["wage_empl_permanent_hours_per_year"] = np.where(
    df["wage_empl_type"] == "Permanent",
    12
    * (df["wage_empl_permanent_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["wage_empl_permanent_hours_per_day"],
    np.nan
)

In [8]:
#SEASONAL/CAUSAL WAGE EMPLOYMENT
df["wage_empl_seasonal_casual_hours_per_year"] = np.where(
    df["wage_empl_type"] == "Seasonal",
    (df["wage_empl_seasonal_casual_rainy_season_duration_in_months_last_12_months"]
     + df["wage_empl_seasonal_casual_dry_season_duration_in_months_last_12_months"])
    * (df["wage_empl_seasonal_casual_duration_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["wage_empl_seasonal_casual_duration_hours_per_day"],
    np.nan
)

In [9]:
#TOTAL YEARLY WORK HOURS
cols = [
    "farm_empl_hours_per_year",
    "self_empl_hours_per_year",
    "wage_empl_permanent_hours_per_year",
    "wage_empl_seasonal_casual_hours_per_year",
]
df["total_work_hours_per_year"] = df[cols].sum(axis=1, min_count=1)

### 2.3 Work hours per year - relative numbers

In [10]:
#WORK TYPE SHARES
# avoid dividing by zero -> treat a total of 0 hours as NaN (undefined share)
total_safe = df["total_work_hours_per_year"].replace(0, np.nan)

df["self_empl_hours_annual_share"] = df["self_empl_hours_per_year"] / total_safe
df["wage_empl_permanent_hours_annual_share"] = df["wage_empl_permanent_hours_per_year"] / total_safe
df["wage_empl_seasonal_casual_hours_annual_share"] = df["wage_empl_seasonal_casual_hours_per_year"] / total_safe

In [11]:
# OFF-FARM WORK SHARE
off_farm_cols = [
    "self_empl_hours_per_year",
    "wage_empl_permanent_hours_per_year",
    "wage_empl_seasonal_casual_hours_per_year",
]

# sum off-farm categories, treating "not applicable" (NaN) as 0 
df["off_farm_hours_per_year"] = df[off_farm_cols].sum(axis=1, min_count=0)

# but if the person has NO work data at all, keep it NaN rather than 0
df.loc[df["total_work_hours_per_year"].isna(), "off_farm_hours_per_year"] = np.nan

# share of total work time that is off-farm
total_safe = df["total_work_hours_per_year"].replace(0, np.nan)
df["off_farm_hours_annual_share"] = df["off_farm_hours_per_year"] / total_safe
df["on_farm_hours_annual_share"] = 1 - df["off_farm_hours_annual_share"]

### 2.4 Hourly wage and annual income - absolute numbers

In [12]:
#FARMING
country_wage_map = {
    "Botswana": feature_engineering.agricultural_wage_per_hour(df, "Botswana",    payment_frequency="Per month",  agriculture_only=False) ,
    "Kenya":    feature_engineering.agricultural_wage_per_hour(df, "Kenya",       payment_frequency="Per day",    agriculture_only=True) ,
    "Namibia":  feature_engineering.agricultural_wage_per_hour(df, "Namibia",     payment_frequency="Per month",  agriculture_only=False) ,
    "Tanzania": feature_engineering.agricultural_wage_per_hour(df, "Tanzania",    payment_frequency="Per day",    agriculture_only=True) ,
    "Zambia":   feature_engineering.agricultural_wage_per_hour(df, "Zambia",      payment_frequency="Per day",    agriculture_only=False) ,
}

condition = df["farm_empl_last_12_months"] == 1

df["farm_empl_wage_per_hour"] = np.where(
    condition,
    df["country"].map(country_wage_map), 
    np.nan
)

df["farm_empl_income_per_year"] = df["farm_empl_wage_per_hour"] * df["farm_empl_hours_per_year"]

In [13]:
#SELF-EMPLOYMENT
input_costs_cols = [
    "self_empl_input_costs_last_30_days",
    "self_empl_labor_costs_last_30_days",
    "self_empl_capital_costs_last_30_days",
]
df["self_empl_input_costs"] = df[input_costs_cols].sum(axis=1, min_count=0)

df["self_empl_wage_per_month"] = df["self_empl_sales_last_30_days"] - df["self_empl_input_costs"]

df["self_empl_hours_per_month"] = np.where(
    df["empl_type"] == "Self-employed/own business",
    (df["self_empl_days_per_week"] * 4)
    * df["self_empl_hours_per_day"],
    np.nan
)

total_safe = df["self_empl_hours_per_month"].replace(0, np.nan)
df["self_empl_wage_per_hour"] = df["self_empl_wage_per_month"] / total_safe

df["self_empl_income_per_year"] = df["self_empl_wage_per_hour"] * df["self_empl_hours_per_year"]

In [14]:
#PERMANENT WAGE EMPLOYMENT
df["wage_empl_permanent_hours_per_month"] = df["wage_empl_permanent_hours_per_year"] / 12

total_safe = df["wage_empl_permanent_hours_per_month"].replace(0, np.nan)
df["wage_empl_permanent_wage_per_hour"] = df["wage_empl_permanent_wage_per_month"] / total_safe

df["wage_empl_permanent_income_per_year"] = df["wage_empl_permanent_wage_per_hour"] * df["wage_empl_permanent_hours_per_year"]

In [15]:
#SEASONAL/CAUSAL WAGE EMPLOYMENT
df["wage_empl_seasonal_casual_wage_per_hour"] = df.apply(feature_engineering.compute_hourly_wage_for_casual_work, axis=1)

df["wage_empl_seasonal_casual_income_per_year"] = df["wage_empl_seasonal_casual_wage_per_hour"] * df["wage_empl_seasonal_casual_hours_per_year"]

In [16]:
#TOTAL YEARLY INCOME
cols = [
    "farm_empl_income_per_year",
    "self_empl_income_per_year",
    "wage_empl_permanent_income_per_year",
    "wage_empl_seasonal_casual_income_per_year",
]
df["total_income_per_year"] = df[cols].sum(axis=1, min_count=1)

### 2.5 Hourly wage and annual income - relative numbers

In [17]:
#WORK TYPE SHARES
# avoid dividing by zero -> treat a total of 0 hours as NaN (undefined share)
total_safe = df["total_income_per_year"].replace(0, np.nan)

df["self_empl_income_annual_share"] = df["self_empl_income_per_year"] / total_safe
df["wage_empl_permanent_income_annual_share"] = df["wage_empl_permanent_income_per_year"] / total_safe
df["wage_empl_seasonal_casual_income_annual_share"] = df["wage_empl_seasonal_casual_income_per_year"] / total_safe

In [18]:
# OFF-FARM WAGE SHARE
off_farm_cols = [
    "self_empl_income_per_year",
    "wage_empl_permanent_income_per_year",
    "wage_empl_seasonal_casual_income_per_year",
]

# sum off-farm categories, treating "not applicable" (NaN) as 0 
df["off_farm_income_per_year"] = df[off_farm_cols].sum(axis=1, min_count=0)

# but if the person has NO work data at all, keep it NaN rather than 0
df.loc[df["total_income_per_year"].isna(), "off_farm_income_per_year"] = np.nan

# share of total work time that is off-farm
total_safe = df["total_income_per_year"].replace(0, np.nan)
df["off_farm_income_annual_share"] = df["off_farm_income_per_year"] / total_safe
df["on_farm_income_annual_share"] = 1 - df["off_farm_income_annual_share"] 

### 2.5 How much more profitable is a job/own business compared to farming?

In [19]:
#This is only calculated for the hybrid workers
total_safe = df["farm_empl_wage_per_hour"].replace(0, np.nan)

df["self_empl_wage_premium_vs_farm"] = df["self_empl_wage_per_hour"] / total_safe
df["wage_empl_permanent_wage_premium_vs_farm"] = df["wage_empl_permanent_wage_per_hour"] / total_safe
df["wage_empl_seasonal_casual_wage_premium_vs_farm"] = df["wage_empl_seasonal_casual_wage_per_hour"] / total_safe

### 2.6 Main income use shares

In [20]:
employment_types    =  ["self_empl", "wage_empl_permanent", "wage_empl_seasonal_casual"]
spending_categories =  ["invest_in_own_business", "food", "education", "health", "housing_furniture", "transportation", "entertainment"]

df = feature_engineering.collapse_main_income_use(df, employment_types, spending_categories)

### 2.6 Self-employment: obstacles + financial constraints + loan source

In [21]:
#SELF-EMPLOYMENT OBSTACLES
cols = [c for c in df.columns if c.startswith('self_empl_obstacle_')]
df = feature_engineering.count_ones(df, cols, "self_empl_obstacle_index")

In [22]:
#SELF-EMPLOYMENT FINANCIAL CONSTRAINTS -> it will only give me a dummy: 1 for "yes there are constraints", 0 for "no contraints". 
cols = [c for c in df.columns if c.startswith('self_empl_three_main_finance_constr_')]
df = feature_engineering.count_ones(df, cols, "self_empl_finance_constr_index")

In [23]:
#SELF-EMPLOYMENT LOAN SOURCES
cols = [c for c in df.columns if c.startswith('self_empl_loan_source_')]
df = feature_engineering.count_ones(df, cols, "self_empl_loan_source_index")

### 2.7 Self-employment: push and pull motivations


In [24]:
push_cols = ['self_empl_motiv_unemployment',
             'self_empl_motiv_insuff_income_from_farming',
             'self_empl_motiv_insuff_income_from_agr_job',
             'self_empl_motiv_insuff_income_from_non_agr_job']
df = feature_engineering.count_ones(df, push_cols, "self_empl_push_motivation_index")


pull_cols = ['self_empl_motiv_previous_experience',
             'self_empl_motiv_others_are_successful',
             'self_empl_motiv_believe_in_success',
             'self_empl_motiv_inherited_business']
df = feature_engineering.count_ones(df, pull_cols, "self_empl_pull_motivation_index")

### 2.8 Number of working (adult) age household members


In [25]:
#TOTAL NUMBER OF WORKING AGE ADULTS
df["hh_members_count"] = df.groupby(["country", "interview_key"]).transform("size")

#SHARE OF WORKING AGE PEOPLE BY EMPLOYMENT CATEGORY
# for a household with missing empl_category values, the three share columns will sum to less than 1 
# — the "missing" fraction is implicitly the share of members whose category is unknown
categories = ["only_farm", "hybrid", "fully_off_farm"]
no_data_mask = df["empl_category"].isna() & df["farm_empl_last_12_months"].isna()

for cat in categories:
    df[f"hh_members_{cat}_share"] = (
        df.groupby(["country", "interview_key"])["empl_category"]
          .transform(lambda s, cat=cat: (s == cat).sum())
        / df["hh_members_count"]
    )
    df.loc[no_data_mask, f"hh_members_{cat}_share"] = np.nan

### 2.7 Employment cleanup

In [26]:
cols_to_drop = [
    "self_empl_duration_dry_season_in_months_last_12_months",
    "self_empl_duration_rainy_season_in_months_last_12_months",
    "self_empl_days_per_week",
    "self_empl_hours_per_day",
    "self_empl_sales_last_30_days",
    "self_empl_main_use_invest_in_own_business",
    "self_empl_main_use_food",
    "self_empl_main_use_education",
    "self_empl_main_use_health",
    "self_empl_main_use_housing_furniture",
    "self_empl_main_use_transportation",
    "self_empl_main_use_entertainment",
    "self_empl_input_costs_last_30_days",
    "self_empl_labor_costs_last_30_days",
    "self_empl_capital_costs_last_30_days",
    "self_empl_motiv_previous_experience",
    "self_empl_motiv_others_are_successful",
    "self_empl_motiv_believe_in_success",
    "self_empl_motiv_unemployment",
    "self_empl_motiv_insuff_income_from_farming",
    "self_empl_motiv_insuff_income_from_agr_job",
    "self_empl_motiv_insuff_income_from_non_agr_job",
    "self_empl_motiv_inherited_business",
    "self_empl_obstacle_taxes_regulation",
    "self_empl_obstacle_financing",
    "self_empl_obstacle_political_instability",
    "self_empl_obstacle_inflation",
    "self_empl_obstacle_infrastructure",
    "self_empl_obstacle_organised_crime",
    "self_empl_obstacle_street_crime",
    "self_empl_obstacle_corruption",
    "self_empl_obstacle_no_purchasing_power",
    "self_empl_obstacle_racial_discrimination",
    "self_empl_obstacle_no_land_access",
    "self_empl_three_main_finance_constr_high_int_rate",
    "self_empl_three_main_finance_constr_no_long_term_loan",
    "self_empl_three_main_finance_constr_no_collateral",
    "self_empl_three_main_finance_constr_paperwork",
    "self_empl_three_main_finance_constr_credit_info",
    "self_empl_three_main_finance_constr_connections",
    "self_empl_three_main_finance_constr_bank_lacks_money",
    "self_empl_three_main_finance_constr_no_export_finance",
    "self_empl_three_main_finance_constr_no_equity",
    "self_empl_three_main_finance_constr_no_leasing",
    "self_empl_three_main_finance_constr_no_foreign_banks",
    "self_empl_three_main_finance_constr_corruption",
    "self_empl_loan_source_retained_earnings",
    "self_empl_loan_source_local_banks",
    "self_empl_loan_source_family_friends",
    "self_empl_loan_source_supplier_credit",
    "self_empl_loan_source_sale_of_stock",
    "self_empl_loan_source_foreign_banks",
    "self_empl_loan_source_develop_finance",
    "self_empl_loan_source_moneylenders",
    "wage_empl_permanent_wage_per_month",
    "wage_empl_permanent_main_use_invest_in_own_business",
    "wage_empl_permanent_main_use_food",
    "wage_empl_permanent_main_use_education",
    "wage_empl_permanent_main_use_health",
    "wage_empl_permanent_main_use_housing_furniture",
    "wage_empl_permanent_main_use_transportation",
    "wage_empl_permanent_main_use_entertainment",
    "wage_empl_permanent_days_per_week",
    "wage_empl_permanent_hours_per_day",
    "wage_empl_seasonal_casual_payment_frequency",
    "wage_empl_seasonal_casual_wage_per_interval",
    "wage_empl_seasonal_casual_main_use_invest_in_own_business",
    "wage_empl_seasonal_casual_main_use_food",
    "wage_empl_seasonal_casual_main_use_education",
    "wage_empl_seasonal_casual_main_use_health",
    "wage_empl_seasonal_casual_main_use_housing_furniture",
    "wage_empl_seasonal_casual_main_use_transportation",
    "wage_empl_seasonal_casual_main_use_entertainment",
    "wage_empl_seasonal_casual_rainy_season_duration_in_months_last_12_months",
    "wage_empl_seasonal_casual_dry_season_duration_in_months_last_12_months",
    "wage_empl_seasonal_casual_duration_days_per_week",
    "wage_empl_seasonal_casual_duration_hours_per_day",
    "farm_empl_cash_crops_duration_rainy_season_in_months_last_12_months",
    "farm_empl_cash_crops_duration_dry_season_in_months_last_12_months",
    "farm_empl_cash_crops_days_per_week",
    "farm_empl_cash_crops_hours_per_day",
    "farm_empl_food_crops_duration_rainy_season_in_months_last_12_months",
    "farm_empl_food_crops_duration_dry_season_in_months_last_12_months",
    "farm_empl_food_crops_days_per_week",
    "farm_empl_food_crops_hours_per_day",
    "farm_empl_livestock_duration_rainy_season_in_months_last_12_months",
    "farm_empl_livestock_duration_dry_season_in_months_last_12_months",
    "farm_empl_livestock_days_per_week",
    "farm_empl_livestock_hours_per_day",
    "farm_empl_main_use_invest_in_own_business",
    "farm_empl_main_use_food",
    "farm_empl_main_use_education",
    "farm_empl_main_use_health",
    "farm_empl_main_use_housing_furniture",
    "farm_empl_main_use_transportation",
    "farm_empl_main_use_entertainment",
    'self_empl_input_costs',
    'self_empl_wage_per_month',
    'self_empl_hours_per_month',
    'wage_empl_permanent_hours_per_month',
]

df = df.drop(columns=cols_to_drop, errors="ignore")


## 3. Household optimism and sentiment index

In [27]:
optimism_cols = [
    "my_life_course_depends_on_me",
    "my_plans_will_work",
    "I_can_shape_my_future_positively",
    "I_am_optimistic_about_my_future",
    "I_am_optimistic_about_my_familys_future",
]

df["hh_optimism_index"] = df[optimism_cols].mean(axis=1)
df["hh_sentiment_index"] = df[["hh_optimism_index", "worry_about_job_loss_or_economic_livelihood"]].mean(axis=1)
df = df.drop(columns=optimism_cols, errors="ignore")

## 4. Livestock household features

### 4.1 Grazing land challenges

In [28]:
cols = ['grazing_land_challenges_little_gras',
        'grazing_land_challenges_prosopis_parthenium',
        'grazing_land_challenges_other_pastoralists',
        'grazing_land_challenges_ethnic_conflict',
        'grazing_land_challenges_tension_conflict',
        'grazing_land_challenges_theft',
        'grazing_land_challenges_raiding',
        'grazing_land_challenges_no_water',
        'grazing_land_challenges_too_far',
        'grazing_land_challenges_expensive']
df = feature_engineering.count_ones(df, cols, "grazing_land_challenges_index")
df = df.drop(cols, axis=1)


### 4.2 Expenses and profit

In [29]:
#LIVESTOCK TOTAL EXPENSES
expenses_cols = ['livestock_exp_feed_fodder',
                'livestock_exp_rent_gazing_land',
                'livestock_exp_veterinary_services',
                'livestock_exp_shelter',
                'livestock_exp_hired_labor']
df["livestock_exp_last_12_months"] = df[expenses_cols].sum(axis=1, min_count=1)


#LIVESTOCK TOTAL PROFIT
df["livestock_profit_last_12_months"] = df["livestock_income_last_12_months"] - df["livestock_exp_last_12_months"]

#LIVESTOCK COST RATIO
total_safe = df["livestock_income_last_12_months"].replace(0, np.nan)
df["livestock_cost_ratio"] = df["livestock_exp_last_12_months"] / total_safe

df = df.drop(expenses_cols, axis=1)

## 5. Crop farming household features

### 5.1 Expenses and profit

In [30]:
#CROP TOTAL EXPENSES
expenses_cols = ['crop_exp_seeds_last_12_months',
                'crop_exp_fertilizer_last_12_months',
                'crop_exp_pesticide_last_12_months',
                'crop_exp_machinery_last_12_months',
                'crop_exp_hired_labor_last_12_months',
                'crop_exp_land_rental_last_12_months',
                'crop_exp_transport_last_12_months',
                'crop_exp_other_last_12_months']
df["crop_exp_last_12_months"] = df[expenses_cols].sum(axis=1, min_count=1)

#CROP TOTAL PROFIT
df["crop_profit_last_12_months"] = df["crop_sale_revenue_last_12_months"] - df["crop_exp_last_12_months"]

#CROP COST RATIO
total_safe = df["crop_sale_revenue_last_12_months"].replace(0, np.nan)
df["crop_cost_ratio"] = df["crop_exp_last_12_months"] / total_safe

df = df.drop(expenses_cols, axis=1)

### 5.1 Revenue and profit per acre

In [31]:
cols = ["land_size_cropland_acres", "land_size_fallow_acres", "land_size_agroforestry_forestry_acres"]
df["land_size_agriculture_acres"] = df[cols].sum(axis=1, min_count=1)

#CROP REVENUE PER ACRE
total_safe = df["land_size_agriculture_acres"].replace(0, np.nan)
df["crop_sale_revenue_per_acre"] = df["crop_sale_revenue_last_12_months"] / total_safe

#CROP PROFIT PER ACRE
total_safe = df["land_size_agriculture_acres"].replace(0, np.nan)
df["crop_profit_per_acre"] = df["crop_profit_last_12_months"] / total_safe

df = df.drop("land_size_agriculture_acres", axis=1)

## 6. Currency conversion

In [32]:
conversion_cols = [
    "livestock_income_last_12_months",
    'livestock_revenue_sold',
    'livestock_exp_last_12_months',
    'livestock_profit_last_12_months',
    'crop_sale_revenue_last_12_months',
    'crop_exp_last_12_months',
    'crop_profit_last_12_months',
    'crop_sale_revenue_per_acre',
    'crop_profit_per_acre',
    'asset_value',
    'other_income_amount_annual',
    'remittance_amount_sent_last_12_months',
    'agriculture_loan_amount',
    'farm_empl_wage_per_hour',
    'self_empl_wage_per_hour',
    'wage_empl_permanent_wage_per_hour',
    'wage_empl_seasonal_casual_wage_per_hour',
    'farm_empl_income_per_year',
    'self_empl_income_per_year',
    'wage_empl_permanent_income_per_year',
    'wage_empl_seasonal_casual_income_per_year',
    'total_income_per_year',
    'off_farm_income_per_year']


df = feature_engineering.convert_currency(df, conversion_cols, "country", mappings.usd_exchange_rates )

## 7. Write features to data/processed folder

In [33]:
#merge.to_csv(config.PROCESSED_DATA_DIR / "clean_data.csv", index=False)
df.to_csv(config.PROCESSED_DATA_DIR / "features_data.csv", index=False, encoding="utf-8-sig")